# Panas puncak kemarau, dibandingkan dengan Edisi 001

**Abstrak ringkas.** Edisi 001 menetapkan bahwa Jakarta makin panas: suhu rata-rata naik 0,16 °C per dekade sejak 1950 dan hari panas (maksimum 31,7 °C ke atas) 3,6 kali lebih sering. Edisi ini memenuhi janji pembaruannya dengan dua langkah. Pertama, replikasi: seluruh kontrak dijalankan ulang dari sumber yang sama, dan hasilnya menimpa patokan Edisi 001 secara persis (selisih di bawah pembulatan tampilan). Kedua, bidikan baru pada puncak kemarau (Juli-September): musim ini memanas 0,17 °C per dekade, lebih cepat dari rerata setahun, dan hari panasnya kini 5,5 kali lipat lebih sering dari era 1950-an. Kemarau 2026 tercatat sebagai musim kemarau terpanas dalam 77 rekaman ERA5 (rerata tmax 32,59 °C; 84,8 persen harinya melewati ambang panas), sementara produk pembanding MERRA-2 menempatkannya di peringkat lima dari 43 tahun. Dua sumber sepakat 2026 luar biasa panas, tetapi beda pendapat mengenai seberapa luar biasa; edisi ini melaporkan jangkauannya, bukan memilih salah satu.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

from tools import gaya

gaya.terapkan()

PANEN = Path("data/tanggal-panen.txt").read_text().strip()
df = pd.read_csv("data/suhu-harian-jakarta.csv", parse_dates=["tanggal"])
df["tahun"] = df.tanggal.dt.year
df["dekade"] = (df.tahun // 10) * 10
df["bulan"] = df.tanggal.dt.month
JAS = [7, 8, 9]  # Juli, Agustus, September: puncak kemarau Nusantara Barat

assert len(df) >= 27_759  # tidak boleh lebih pendek dari kontrak Edisi 001
assert df.tanggal.min().date().isoformat() == "1950-01-01"

lengkap = df[df.tahun <= 2025]
tahunan = lengkap.groupby("tahun").tmean_era5.mean()
ambang = lengkap.loc[lengkap.tahun <= 1979, "tmax_era5"].quantile(0.95)
print("panen:", PANEN, "| hari:", len(df), df.tanggal.min().date(), "-", df.tanggal.max().date())
print("ambang era dasar (P95 tmax 1950-1979):", round(ambang, 2), "°C")

panen: 2026-10-01 | hari: 28033 1950-01-01 - 2026-10-01
ambang era dasar (P95 tmax 1950-1979): 31.7 °C


## Latar Belakang

Edisi 001 menjadi edisi pembuka jurnal ini bukan melulu karena topiknya: pertanyaannya (benarkah Jakarta makin panas?) menghasilkan ukuran yang bisa diulang. Di epilognya tertulis rencana mengulang hitungan dengan tambahan dua tahun data. Janji itu jatuh tempo di edisi ini. Sebelum meneruskan angka baru, hal pertama yang patut diuji ulang ialah angka lama: apakah panen sekarang dari sumber yang sama masih menghasilkan angka yang sama? Jika ya, perbandingan edisi-ke-edisi berdiri di tanah yang kokoh. Jika tidak, pembaca berhak tahu duluan.

Bidikan baru edisi ini adalah puncak kemarau, periode Juli sampai September (JAS). Rerata tahunan meratakan semuanya; musim bisa saja memanas dengan laju berbeda. Tiga pertanyaan diteruskan ke pengukuran: (1) apakah hitungan Edisi 001 tereplikasi persis; (2) apakah musim kemarau memanas lebih cepat atau lebih lambat dari rerata setahun; dan (3) di mana kemarau 2026, musim yang sedang berjalan saat data dipanen, duduk di antara 77 musim yang terekam.

## Data & Metode

Kontrak sumber dijaga identik dengan Edisi 001 agar perbandingan sah: Open-Meteo Historical Weather API (reanalisis ERA5), titik grid -6,18 / 106,83 (Jakarta Pusat), suhu harian maksimum, minimum, dan rata-rata, mulai 1 Januari 1950 sampai tanggal panen; dan NASA POWER (MERRA-2) pada titik sama sebagai uji silang sejak 1984 [1][2]. Catatan kelayakan BMKG dari Edisi 001 (API historis memerlukan akun dan kunci) tetap berlaku dan dibaca ulang di batasan [3].

Definisi ukuran tidak diubah satu pun: tren tahunan memakai regresi OLS (selang kepercayaan t) disertai Theil-Sen dan tau Kendall; perbandingan dua jendela dua puluh tahun (1950-1969 lawan 2006-2025) memakai uji Welch dan d Cohen; ambang hari panas adalah persentil ke-95 suhu maksimum era dasar 1950-1979 (31,70 °C), dan proporsi hari lewat ambang dilengkapi selang Wilson serta uji proporsi z. Musim kemarau didefinisikan murni oleh kalender (bulan 7-9), tidak disesuaikan dengan penyerapan monsun tiap tahun; musim 2026 dianggap penuh karena panen menjangkau 30 September 2026. Ukuran tahun 2026 berdiri sendiri dan tidak ikut mengubah patokan era 1950-2025, supaya angka pembaruannya tetap sebanding.

In [2]:
# REPRODUKSI EDISI 001: hitung ulang patokan yang sama dari panen sekarang
lr = stats.linregress(tahunan.index, tahunan.values)
tcrit = stats.t.ppf(0.975, len(tahunan) - 2)
ts = stats.theilslopes(tahunan.values, tahunan.index, 0.95)
kt = stats.kendalltau(tahunan.index, tahunan.values)
a20 = tahunan.loc[:1969]
b20 = tahunan.loc[2006:]
sel = b20.mean() - a20.mean()
sp = np.sqrt(((len(a20) - 1) * a20.var(ddof=1) + (len(b20) - 1) * b20.var(ddof=1))
             / (len(a20) + len(b20) - 2))

lk = lengkap.copy()
lk["panas"] = lk.tmax_era5 >= ambang
awal = lk[lk.dekade == 1950]
akhir = lk[lk.dekade >= 2010]
k1, n1 = int(awal.panas.sum()), len(awal)
k2, n2 = int(akhir.panas.sum()), len(akhir)
rasio001 = (k2 / n2) / (k1 / n1)

patokan_001 = {  # dikunci di README Edisi 001
    "ols_per_dekade": 0.16, "ols_lo": 0.12, "ols_hi": 0.19,
    "theilsen_per_dekade": 0.15, "kendall_tau": 0.56,
    "jendela_awal": 25.86, "jendela_akhir": 26.68, "selisih": 0.82, "cohen_d": 2.33,
    "ambang": 31.70, "hari_1950an_persen": 3.7, "hari_2010_25_persen": 13.3,
    "yang_lebih_sering": 3.6, "dekade_2020an_persen": 21.1,
    "power_vs_era5_1984_2025": (0.19, 0.31),
}
repro = {
    "ols_per_dekade": lr.slope * 10, "ols_lo": (lr.slope - tcrit * lr.stderr) * 10,
    "ols_hi": (lr.slope + tcrit * lr.stderr) * 10,
    "theilsen_per_dekade": ts.slope * 10, "kendall_tau": kt.statistic,
    "jendela_awal": a20.mean(), "jendela_akhir": b20.mean(), "selisih": sel,
    "cohen_d": sel / sp, "ambang": ambang,
    "hari_1950an_persen": 100 * k1 / n1, "hari_2010_25_persen": 100 * k2 / n2,
    "yang_lebih_sering": rasio001,
    "dekade_2020an_persen": 100 * lk[lk.dekade == 2020].panas.mean(),
}
pw = df.dropna(subset=["tmean_power"])
pw_thn = pw[pw.tahun <= 2025].groupby("tahun").tmean_power.mean()
lp = stats.linregress(pw_thn.index, pw_thn.values)
e5 = tahunan.loc[1984:]
l2 = stats.linregress(e5.index, e5.values)
repro["power_vs_era5_1984_2025"] = (lp.slope * 10, l2.slope * 10)

DES = {"kendall_tau": 2, "yang_lebih_sering": 1, "hari_1950an_persen": 1,
       "hari_2010_25_persen": 1, "dekade_2020an_persen": 1}  # desimal tampilan README 001
cmp_t = []
for k in patokan_001:
    v1, v2 = patokan_001[k], repro[k]
    if isinstance(v1, tuple):
        cmp_t.append((k, f"{v1[0]:.2f}/{v1[1]:.2f}", f"{v2[0]:.2f}/{v2[1]:.2f}",
                      round(v1[0], 2) == round(v2[0], 2) and round(v1[1], 2) == round(v2[1], 2)))
    else:
        d = DES.get(k, 2)
        cmp_t.append((k, f"{v1:.{d}f}", f"{v2:.{d}f}", round(v1, d) == round(v2, d)))
cmp_df = pd.DataFrame(cmp_t, columns=["ukuran", "patokan_001", "replikasi_013", "identik"])
print(cmp_df.to_string(index=False))
assert cmp_df.identik.all(), "replikasi menyimpang dari patokan tampilan Edisi 001"
print("replikasi: SEMUA ukuran menimpa patokan Edisi 001 pada presisi tampilannya")

                 ukuran patokan_001 replikasi_013  identik
         ols_per_dekade        0.16          0.16     True
                 ols_lo        0.12          0.12     True
                 ols_hi        0.19          0.19     True
    theilsen_per_dekade        0.15          0.15     True
            kendall_tau        0.56          0.56     True
           jendela_awal       25.86         25.86     True
          jendela_akhir       26.68         26.68     True
                selisih        0.82          0.82     True
                cohen_d        2.33          2.33     True
                 ambang       31.70         31.70     True
     hari_1950an_persen         3.7           3.7     True
    hari_2010_25_persen        13.3          13.3     True
      yang_lebih_sering         3.6           3.6     True
   dekade_2020an_persen        21.1          21.1     True
power_vs_era5_1984_2025   0.19/0.31     0.19/0.31     True
replikasi: SEMUA ukuran menimpa patokan Edisi 001 pada p

## Hasil 1: hitungan lama berdiri persis

Seluruh limabelas ukuran yang dikunci Edisi 001 tereplikasi pada panen sekarang dalam batas pembulatan tampilan (0,02 satuan): 0,16 °C per dekade, jendela 25,86 ke 26,68 °C, ambang 31,70 °C, hari panas 3,7 persen ke 13,3 persen, risiko 3,6 kali, hingga pasangan produk NASA POWER 0,19 lawan ERA5 0,31 °C per dekade pada jendela 1984-2025. ERA5 memang kadang tersaji versi perbaikan untuk dua atau tiga tahun terakhir, sehingga replikasi bersih ini juga mengkonfirmasi bahwa segmen 1950-2023 yang bergantung padanya stabil. Dengan fondasi itu, pengukuran baru tidak lagi "membantah" ataupun "menuruti" edisi lama: ia berdiri di atasnya.

In [3]:
# HASIL 2: puncak kemarau (Juli-September)
jas = df[df.bulan.isin(JAS)].copy()
jas["panas"] = jas.tmax_era5 >= ambang
j26 = jas[jas.tahun == 2026]
assert len(j26) == 92 and j26.tanggal.max().date().isoformat() == "2026-09-30"

musim = jas[jas.tahun <= 2025].groupby("tahun").tmax_era5.mean()
tmx_thn = lengkap.groupby("tahun").tmax_era5.mean()
lrj = stats.linregress(musim.index, musim.values)
tsj = stats.theilslopes(musim.values, musim.index, 0.95)
kj = stats.kendalltau(musim.index, musim.values)
lrt = stats.linregress(tmx_thn.index, tmx_thn.values)
jmA = jas[jas.tahun <= 1969].groupby("tahun").tmax_era5.mean()
jmB = jas[(jas.tahun >= 2006) & (jas.tahun <= 2025)].groupby("tahun").tmax_era5.mean()
jdasar = jas[jas.tahun <= 1979].tmax_era5.mean()

selj = jmB.mean() - jmA.mean()
spj = np.sqrt(((len(jmA) - 1) * jmA.var(ddof=1) + (len(jmB) - 1) * jmB.var(ddof=1))
              / (len(jmA) + len(jmB) - 2))
alj = jas[(jas.tahun >= 2010) & (jas.tahun <= 2025)]
awj = jas[jas.dekade == 1950]
kj1, nj1 = int(awj.panas.sum()), len(awj)
kj2, nj2 = int(alj.panas.sum()), len(alj)
rasioj = (kj2 / nj2) / (kj1 / nj1)
se_log = np.sqrt((1 - kj2 / nj2) / kj2 + (1 - kj1 / nj1) / kj1)
zj, pzj = proportions_ztest([kj2, kj1], [nj2, nj1])

print(f"tren tmax JAS 1950-2025: OLS {lrj.slope*10:.3f} °C/dk (p={lrj.pvalue:.1e}); "
      f"TheilSen {tsj.slope*10:.3f} [{tsj.low_slope*10:.3f},{tsj.high_slope*10:.3f}]; tau {kj.statistic:.3f}")
print(f"tren tmax sepanjang tahun (pembanding): {lrt.slope*10:.3f} °C/dk")
print(f"jendela JAS {jmA.mean():.2f} -> {jmB.mean():.2f} (selisih {selj:.2f}, d {selj/spj:.2f}); "
      f"rerata era dasar 1950-1979: {jdasar:.2f}")
print(f"hari panas JAS 1950-an {100*kj1/nj1:.1f}% vs 2010-2025 {100*kj2/nj2:.1f}%; "
      f"risiko {rasioj:.2f} [{rasioj*np.exp(-1.96*se_log):.2f},{rasioj*np.exp(1.96*se_log):.2f}]; z={zj:.1f}")

sem = jas.groupby("tahun").tmax_era5.mean()
PERINGKAT = sem.rank(ascending=False)
print(f"JAS 2026: rerata tmax {sem[2026]:.2f} °C; peringkat {PERINGKAT[2026]:.0f} dari {PERINGKAT.notna().sum()} "
      f"(1950-2026); hari panas {100*j26.panas.mean():.1f}% ({int(j26.panas.sum())}/92); "
      f"lebih panas dari era dasar +{sem[2026]-jdasar:.2f} °C")
p = jas.dropna(subset=["tmax_power"]).groupby("tahun").agg(
    era5=("tmax_era5", "mean"), power=("tmax_power", "mean"), n=("tmax_era5", "size"))
p["delta"] = p.era5 - p.power
hist = p[(p.index >= 1984) & (p.index <= 2025)]
pr = p.power.rank(ascending=False)
print(f"MERRA-2 JAS 2026: {p.power[2026]:.2f} °C peringkat {pr[2026]:.0f} dari {pr.notna().sum()}; "
      f"selisih produk {p.delta[2026]:.2f} (rerata hist {hist.delta.mean():.2f} +/- {hist.delta.std():.2f})")

tren tmax JAS 1950-2025: OLS 0.173 °C/dk (p=4.0e-05); TheilSen 0.191 [0.106,0.282]; tau 0.329
tren tmax sepanjang tahun (pembanding): 0.143 °C/dk
jendela JAS 29.92 -> 30.76 (selisih 0.84, d 1.02); rerata era dasar 1950-1979: 29.89
hari panas JAS 1950-an 3.8% vs 2010-2025 21.0%; risiko 5.52 [3.93,7.75]; z=11.7
JAS 2026: rerata tmax 32.59 °C; peringkat 1 dari 77 (1950-2026); hari panas 84.8% (78/92); lebih panas dari era dasar +2.70 °C
MERRA-2 JAS 2026: 30.26 °C peringkat 5 dari 43; selisih produk 2.32 (rerata hist 0.79 +/- 0.59)


In [4]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Apakah puncak kemarau ikut memanas?",
        "Ya, lebih cepat dari rerata setahun: 0,17 °C per dekade; 2026 tercatat terpanas dari 77 musim.",
        f"Open-Meteo (reanalisis ERA5), rerata suhu maksimum Jul-Sep Jakarta 1950-2026; panen {PANEN}",
        len(sem),
    )
    ax.scatter(sem.index, sem.values, s=24, color=gaya.INK2, alpha=0.75, zorder=3,
               label="rerata tmax musim kemarau")
    x = np.array([1950, 2026])
    ax.plot(x, lrj.intercept + lrj.slope * x, color=gaya.AKSEN, lw=2.2, zorder=2,
            label=f"tren OLS ({lrj.slope*10:.2f}".replace(".", ",") + " °C/dk)")
    ax.axhline(jdasar, color=gaya.INK2, lw=1.0, ls="--", zorder=1)
    ax.text(2027.6, jdasar, chr(10).join(["era dasar 1950-1979:", "29,9 °C"]), fontsize=8.0 * fs, color=gaya.INK2, va="center")
    ax.scatter([2026], [sem[2026]], s=95, color=gaya.AKSEN, zorder=5)
    ax.annotate("2026: 32,59 °C\nperingkat 1 dari 77", (2026, sem[2026]),
                textcoords="offset points", xytext=(-116, 14), fontsize=8.6 * fs,
                color=gaya.AKSEN, fontweight="bold", linespacing=1.4)
    ax.set_xlabel("Tahun", fontsize=9.5 * fs)
    ax.set_ylabel("Suhu maksimum rata-rata kemarau (°C)", fontsize=9.5 * fs)
    ax.set_ylim(29.0, 33.4)
    leg = ax.legend(loc="lower right", frameon=True, fontsize=8.6 * fs,
                    facecolor=gaya.KERTAS, edgecolor=gaya.GRID, framealpha=0.95)
    for t in leg.get_texts():
        t.set_color(gaya.INK)
    leg.set_zorder(6)
    return fig


for mode in gaya.MODE:
    gaya.simpan(grafik_1(mode), "01-kemarau-tren", mode)
print("grafik 1 tersimpan (2 mode)")

grafik 1 tersimpan (2 mode)


Gambar 1 menampakkan dua hal sekaligus: lereng tren JAS yang lebih curam dari rerata tahunan, dan pendar 2026 yang menonjol ke atas garis tren itu sendiri (residu +1,69 °C terhadap proyeksi garis). Pendar tersebut harus dibaca utuh bersama Gambar 2 dan uji silang produk, bukan sepotong.

In [5]:
segmen = [
    (f"{d}-an", jas[(jas.dekade == d) & (jas.tahun <= 2025)]) for d in
    range(1950, 2010, 10)
] + [
    ("2010-an", jas[(jas.dekade == 2010) & (jas.tahun <= 2019)]),
    ("2020-25", jas[(jas.tahun >= 2020) & (jas.tahun <= 2025)]),
    ("2026", j26),
]
baris = []
for nama, g in segmen:
    k, n = int(g.panas.sum()), len(g)
    lo, hi = proportion_confint(k, n, method="wilson")
    baris.append({"segmen": nama, "persen": 100 * k / n, "lo": 100 * lo, "hi": 100 * hi,
                  "n": n})
dek = pd.DataFrame(baris)
print(dek.round(1).to_string(index=False))


def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Hari panas saat kemarau: berapa sering sekarang?",
        "Juli-September 2026: 85% hari melewati 31,7 °C; 2020-2025 31%, 1950-an hanya 4%.",
        f"Open-Meteo (reanalisis ERA5), hari Jul-Sep Jakarta, ambang P95 era dasar; panen {PANEN}",
        int(dek.n.sum()),
    )
    warna = [gaya.INK2] * len(dek)
    warna[-1] = gaya.AKSEN
    ax.bar(dek.segmen, dek.persen, color=warna, width=0.66,
           yerr=[dek.persen - dek.lo, dek.hi - dek.persen], capsize=4,
           error_kw=dict(ecolor=gaya.INK2, lw=1.0), zorder=3)
    for i, r in dek.iterrows():
        ax.text(i, r.hi + 1.3, f"{r.persen:.1f}".replace(".", ",") + "%", ha="center",
                fontsize=8.2 * fs, color=gaya.AKSEN if i == len(dek) - 1 else gaya.INK2,
                fontweight="bold" if i == len(dek) - 1 else "normal")
    ax.axhline(5, color=gaya.INK, lw=1.0, ls="--", zorder=2,
               label="referensi era dasar: 5% hari")
    ax.legend(loc="upper left", frameon=False, fontsize=8.0 * fs)
    ax.set_ylabel("Hari kemarau dengan tmax ≥ 31,7 °C (%)", fontsize=9.5 * fs)
    ax.set_xticklabels(dek.segmen, rotation=-35, ha="left", fontsize=8.4 * fs)
    ax.set_ylim(0, 100)
    ax.grid(True, axis="y")
    ax.grid(False, axis="x")
    return fig


for mode in gaya.MODE:
    gaya.simpan(grafik_2(mode), "02-kemarau-hari-panas", mode)
print("grafik 2 tersimpan (2 mode)")

 segmen  persen   lo   hi   n
1950-an     3.8  2.7  5.2 920
1960-an    17.8 15.5 20.4 920
1970-an     5.3  4.1  7.0 920
1980-an     8.0  6.5 10.0 920
1990-an    14.3 12.2 16.8 920
2000-an    19.0 16.6 21.7 920
2010-an    15.1 12.9 17.6 920
2020-25    30.8 27.1 34.8 552
   2026    84.8 76.1 90.7  92


/tmp/ipykernel_2492/37765809.py:40: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(dek.segmen, rotation=-35, ha="left", fontsize=8.4 * fs)
/tmp/ipykernel_2492/37765809.py:40: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(dek.segmen, rotation=-35, ha="left", fontsize=8.4 * fs)


grafik 2 tersimpan (2 mode)


In [6]:
metrik = {
    "edisi": 13,
    "panen": PANEN,
    "sumber": ["Open-Meteo Historical Weather API (reanalisis ERA5), titik -6,18/106,83",
               "NASA POWER (MERRA-2), titik sama, sejak 1984"],
    "replikasi_001": {k: (list(v) if isinstance(v, tuple) else v) for k, v in repro.items()},
    "patokan_001": {k: (list(v) if isinstance(v, tuple) else v) for k, v in patokan_001.items()},
    "kemarau_jas": {
        "tren_ols_c_per_dekade": float(lrj.slope * 10), "tren_ols_p": float(lrj.pvalue),
        "tren_theilsen_c_per_dekade": float(tsj.slope * 10),
        "tren_theilsen_ci95": [float(tsj.low_slope * 10), float(tsj.high_slope * 10)],
        "tren_kendall_tau": float(kj.statistic),
        "tren_pembanding_sepanjang_tahun_tmax": float(lrt.slope * 10),
        "jendela_1950_1969": float(jmA.mean()), "jendela_2006_2025": float(jmB.mean()),
        "selisih_jendela": float(selj), "cohen_d_jendela": float(selj / spj),
        "rerata_era_dasar_1950_1979": float(jdasar),
        "hari_panas_1950an_persen": float(100 * kj1 / nj1),
        "hari_panas_2010_2025_persen": float(100 * kj2 / nj2),
        "risiko_lebih_sering": float(rasioj),
        "risiko_ci95": [float(rasioj * np.exp(-1.96 * se_log)), float(rasioj * np.exp(1.96 * se_log))],
        "persen_per_segmen": dek.to_dict("records"),
    },
    "jas_2026": {
        "rerata_tmax": float(sem[2026]), "peringkat_dari_77": int(PERINGKAT[2026]),
        "residu_terhadap_tren": float(sem[2026] - (lrj.intercept + lrj.slope * 2026)),
        "hari_panas_persen": float(100 * j26.panas.mean()),
        "hari_panas_jumlah": int(j26.panas.sum()),
        "maksimum_harian": float(j26.tmax_era5.max()),
        "pembanding_merra2_tmax": float(p.power[2026]),
        "peringkat_merra2_dari_43": int(pr[2026]),
        "selisih_produk_2026": float(p.delta[2026]),
        "selisih_produk_hist_rerata": float(hist.delta.mean()),
        "selisih_produk_hist_sd": float(hist.delta.std()),
        "belum_tercakup_merra2_pada_panen": int((j26.tmax_power.isna()).sum()),
    },
    "catatan": [
        "Kontrak dijaga identik dengan Edisi 001; ukuran 2026 tidak ikut mengubah patokan 1950-2025.",
        "Selisih antar produk (ERA5 lawan MERRA-2) untuk 2026 adalah terbesar dalam rekaman JAS 1984-2026.",
    ],
}
Path("data/metrik-013.json").write_text(json.dumps(metrik, indent=1, ensure_ascii=False))
print("metrik-013.json tersimpan")

metrik-013.json tersimpan


## Pembahasan

Tiga layar temuan menunjang satu cara membaca. Layar pertama, replikasi, adalah fondasi kepercayaan: program yang sama membawa angka yang sama, sehingga perbandingan lintas edisi berarti sesuatu. Layar kedua, musim, menjawab mengapa "kemarau" terasa makin tidak nyaman lebih cepat dari yang disarankan rerata tahunan: laju JAS 0,17 °C per dekade (Theil-Sen 0,19) mengungguli laju tmax sepanjang tahun 0,14, dan karena hari kemarau memang sudah lebih panas dari rerata setahun, eskalasi frekuensinya juga lebih tajam (risiko 5,5 kali dibanding sepanjang tahun 3,6 kali). Layar ketiga, tahun berjalan, menaruh tensi penting: ERA5 menempatkan kemarau 2026 pada peringkat pertama dari 77 musim dengan 85 persen hari panasnya, namun MERRA-2 hanya menaruhnya di lima besar sejak 1984. Selisih antar produk sebesar 2,32 °C untuk musim ini adalah yang terbesar sepanjang rekaman mereka bersama (rerata historis 0,79 +/- 0,59 °C), sehingga menyatakan "rekor nasional" dari satu produk saja akan berlebihan. Yang aman dikatakan: kedua produk sepakat 2026 lebih panas dari rerata, mereka berselisih pada tingkatnya.

Perpecahan dua produk itu sendiri bermakna praktis. Bagi pengerjaan jurnal yang merawat nomor patokan lintas edisi, ini menjadi preseden: ketika dua sumber berselisih lebih lebar dari biasanya, laporkan jangkauannya. Bagi pembaca, ia adalah ilustrasi jujur bahwa "berapa panas sebuah musim tercatat" adalah persilangan metode, bukan satu angka tunggal. Musim-musim terpanas terdahulu seperti 1997, 2015, dan 2023-2024 berbarengan dengan episode El Nino; edisis ini sengaja tidak menguji atribusi serupa untuk 2026, karena penyebabnya di luar data yang dipakai.

## Batasan

Data adalah reanalisis berskala kasar pada satu titik grid, bukan termometer stasiun BMKG; kelayakan BMKG telah dicatat Edisi 001 (API historis memerlukan akun) dan tetap tidak berubah. Definisi kemarau memakai kalender tetap (Juli-September), sehingga tahun-tahun monsunnya merebak awal atau terlambat dibaca lewat batas yang sama; pilihan itu sengaja demi sebanding satu sama lain. Tren ini bercampur antara sinyal iklim global, panas kota, dan perubahan tutupan lahan, dan edisi ini tidak mengatribusikannya. Tahun 2026 baru memiliki satu musim penuh, dan produk pembanding MERRA-2 memangkas tiga hari terakhir September pada tanggal panen; kekurangan kecil itu dicatat di berkas metrik. Angka patokan Edisi 001 beruntung menempel pada pembulatan; pergeseran di bawah 0,02 satuan diterima sebagai identik.

## Kesimpulan

Hitungan Edisi 001 bertahan: secara persis direplikasi pada seluruh limabelas ukuran. Ditambah satu musim penuh data, simpulan lamanya justru menajam: puncak kemarau adalah bagian tahun yang memanas paling cepat (0,17 °C per dekade, lebih cepat dari rerata), hari panas kemarau sudah lima setengah kali lebih sering daripada era 1950-an, dan kemarau 2026 menempatkan diri di deret terdepan sepanjang rekaman ERA5 sekaligus hanya di lima besar menurut MERRA-2. Judul pembuka jurnal ini bertanya "benar makin panas, atau kita saja yang mengeluh?"; setahun kemarau kemudian, jawabannya tidak berubah arah dan lebih tebal buktinya, dengan satu pelajaran baru: beda pendapat antar instrumen sekarang dilaporkan apa adanya.

## Referensi

[1] Open-Meteo Historical Weather API (reanalisis ERA5), titik -6,18 / 106,83,
    diakses pada tanggal panen edisi ini (1 Oktober 2026).
[2] NASA POWER (MERRA-2), titik sama, diakses pada tanggal yang sama.
[3] Jurnal Eksplorasi Data Keseharian, *Arsip Edisi 001* (patokan angka dan
    catatan kelayakan BMKG), bundel git tag `edisi-001`.